# Итоговый проект: разработка гибридной системы рекомендаций книг с нейросетевыми моделями

## Введение

Итоговый проект по разработке гибридной системы рекомендаций книг нацелен на создание универсальной и эффективной платформы, объединяющей классические методы рекомендаций с современными нейросетевыми архитектурами. Основная задача — повысить качество рекомендаций за счет использования разнообразных моделей и стратегий, а также их последующего объединения в единую систему с возможностью дальнейшей оптимизации и расширения.

### Предыдущая работа и базовые модели

В основу проекта заложен предварительный анализ и реализованные модели из домашнего задания №1. В ходе предыдущей работы были созданы и протестированы следующие подходы:

- модель популярности,
- контентные рекомендации (на базе схожести по тегам и названиям),
- коллаборативная фильтрация (Item-Based),
- матричная факторизация с помощью SVD.

Полученные метрики (Precision@K, Recall@K, nDCG@K) позволяют объективно сравнивать эффективность каждого метода.

## Цели и задачи проекта

Создать улучшенную гибридную систему, которая объединит все перечисленные подходы с добавлением нейросетевых методов для повышения релевантности рекомендаций.

## Импортируем необходимые библиотеки

In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import normalize, StandardScaler, MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.decomposition import TruncatedSVD
import pandas as pd
from collections import Counter, defaultdict
from surprise import Dataset, Reader, SVD
from collections import Counter, defaultdict
from itertools import product
from sklearn.model_selection import train_test_split

 ### 1. Улучшенная подготовка данных

In [2]:
train_df = pd.read_csv('data/train_df.csv')
ratings_df = pd.read_csv('data/ratings.csv')
literature = pd.read_csv('data/books.csv')
literature_keywords = pd.read_csv('data/book_tags.csv')
keywords = pd.read_csv('data/tags.csv')
test_df = pd.read_csv('data/test_df.csv')

#### Признаки пользователей

In [3]:
user_features = train_df.groupby('user_id').agg(
    user_rating_count=('rating', 'count'),
    user_rating_mean=('rating', 'mean'),
    user_rating_std=('rating', 'std')
).reset_index()

user_features['user_rating_std'] = user_features['user_rating_std'].fillna(0)

user_features['user_rating_count_log'] = np.log1p(user_features['user_rating_count'])

scaler = MinMaxScaler()
user_features['user_rating_count_log_scaled'] = scaler.fit_transform(
    user_features[['user_rating_count_log']]
).ravel()

user_features[['user_id', 'user_rating_count', 'user_rating_mean', 
                     'user_rating_std', 'user_rating_count_log_scaled']].head(5)


,user_id,user_rating_count,user_rating_mean,user_rating_std,user_rating_count_log_scaled
0,1,2,3.5,0.707107,0.092398
1,2,2,4.0,0.000000,0.092398
2,3,1,1.0,0.000000,0.000000
3,4,2,3.5,2.121320,0.092398
4,5,4,4.0,0.816497,0.208805


### Признаки книг

In [4]:
book_stats = ratings_df.groupby('book_id')['rating'].agg(
    avg_rating='mean',
    total_reviews='count',
    rating_std_dev='std'
).fillna(0)  

book_stats['total_reviews_log'] = np.log1p(book_stats['total_reviews']) 

features_to_scale = ['avg_rating', 'total_reviews_log', 'rating_std_dev']


scaler = MinMaxScaler()
book_stats[features_to_scale] = scaler.fit_transform(book_stats[features_to_scale])


book_stats['book_reliability_score'] = (
    book_stats['avg_rating'] * 
    book_stats['total_reviews_log'] / 
    book_stats['total_reviews_log'].max() 
)

book_stats[['avg_rating', 'total_reviews', 'rating_std_dev', 
                  'total_reviews_log', 'book_reliability_score']].head(5)

,avg_rating,total_reviews,rating_std_dev,total_reviews_log,book_reliability_score
book_id,,,,,
1,0.797140,100,0.339228,1.0,0.797140
2,0.786651,100,0.457906,1.0,0.786651
3,0.395074,100,0.803852,1.0,0.395074
4,0.874057,100,0.295224,1.0,0.874057
5,0.674772,100,0.485431,1.0,0.674772


#### Признаки взаимодействий

In [5]:
literature_for_merge = literature[['book_id', 'original_title']].dropna().copy()
literature_for_merge['original_title'] = literature_for_merge['original_title'].astype(str).str.strip()

books_tags_merged = (
    literature_keywords
    .merge(literature_for_merge, left_on='goodreads_book_id', right_on='book_id', how='inner')
    .merge(keywords[['tag_id', 'tag_name']], on='tag_id', how='inner')
    .drop_duplicates(['book_id', 'tag_name'])
)

tag_profiles = (
    books_tags_merged
    .groupby('book_id')['tag_name']
    .apply(lambda tags: ' '.join(sorted(set(str(t) for t in tags if pd.notna(t)))))
    .reset_index()
)

complete_profiles = (
    literature_for_merge
    .merge(tag_profiles, on='book_id', how='left')
    .fillna({'tag_name': ''})
)

complete_profiles['search_text'] = (
    complete_profiles['original_title'] + ' ' + complete_profiles['tag_name']
).str.lower().str.replace(r'\s+', ' ', regex=True).str.strip()

complete_profiles = complete_profiles[complete_profiles['search_text'].str.len() > 0].copy()

bookid_to_index = {book_id: idx for idx, book_id in enumerate(complete_profiles['book_id'])}
index_to_bookid = {idx: book_id for book_id, idx in bookid_to_index.items()}

print(f"Обработано {len(complete_profiles)} уникальных книг.")

# TF-IDF векторизация
vectorizer = TfidfVectorizer(
    max_features=30_000,
    stop_words='english',
    ngram_range=(1, 3),
    min_df=2,
    max_df=0.95,
    lowercase=True,
    strip_accents='unicode',
    token_pattern=r'\b[a-zA-Z]{2,}\b'
)

tfidf_matrix = vectorizer.fit_transform(complete_profiles['search_text'])
print(f"TF-IDF матрица создана: {tfidf_matrix.shape}")

# Функция средний рейтинг пользователя
def user_mean_rating(user_id):
    user_ratings = train_df[train_df['user_id'] == user_id]['rating']
    return float(user_ratings.mean()) if len(user_ratings) > 0 else 3.0

# Построение профилей пользователей
user_profiles = {}

for user_id, group in tqdm(train_df.groupby('user_id'), desc="Построение профилей пользователей"):
    valid_books = [b for b in group['book_id'] if b in bookid_to_index]
    if not valid_books:
        continue
    idxs = [bookid_to_index[b] for b in valid_books]
    user_vec = tfidf_matrix[idxs].mean(axis=0)  

    # Безопасное преобразование в 1D numpy array
    if hasattr(user_vec, 'toarray'):
        user_vec = user_vec.toarray().ravel()  
    else:
        user_vec = np.asarray(user_vec).ravel()

    # Нормализация L2
    user_vec = normalize([user_vec], norm='l2').ravel() 
    user_profiles[user_id] = user_vec

print(f"Профили построены для {len(user_profiles)} пользователей.")

# Функция схожести
def user_book_sim(user_id, book_id):
    if user_id not in user_profiles or book_id not in bookid_to_index:
        return 0.0
    user_vec = user_profiles[user_id]
    book_idx = bookid_to_index[book_id]
    book_vec = tfidf_matrix[book_idx].toarray().ravel()
    return float(np.dot(user_vec, book_vec))

# Создание признаков взаимодействия
train_pairs = train_df[['user_id', 'book_id']].copy()

# Векторизованное вычисление схожести
train_pairs['similarity'] = train_pairs.apply(lambda row: user_book_sim(row['user_id'], row['book_id']), axis=1)

# Средний рейтинг пользователя
train_pairs['user_mean_rating'] = train_pairs['user_id'].map(user_mean_rating)
train_pairs['rating_deviation'] = train_df['rating'] - train_pairs['user_mean_rating']

# Статистики
user_total_ratings = train_df.groupby('user_id')['book_id'].count().to_dict()
book_total_ratings = train_df.groupby('book_id')['rating'].count().to_dict()

train_pairs['user_total_ratings'] = train_pairs['user_id'].map(user_total_ratings).fillna(0)
train_pairs['book_total_ratings'] = train_pairs['book_id'].map(book_total_ratings).fillna(0)

# Логарифмы для устойчивости
train_pairs['user_total_ratings_log'] = np.log1p(train_pairs['user_total_ratings'])
train_pairs['book_total_ratings_log'] = np.log1p(train_pairs['book_total_ratings'])

# Нормализация отклонения
scaler_deviation = StandardScaler()
train_pairs['rating_deviation_scaled'] = scaler_deviation.fit_transform(train_pairs[['rating_deviation']])

# Окончательный датасет признаков взаимодействия
interaction_features = train_pairs[
    ['user_id', 'book_id', 'similarity', 'rating_deviation', 'rating_deviation_scaled',
     'user_total_ratings_log', 'book_total_ratings_log']
].copy()

interaction_features.head()

Обработано 9415 уникальных книг.
TF-IDF матрица создана: (9415, 30000)


Построение профилей пользователей: 100%|███████████████████████████████████████| 53424/53424 [00:27<00:00, 1965.19it/s]


Профили построены для 25524 пользователей.


,user_id,book_id,similarity,rating_deviation,rating_deviation_scaled,user_total_ratings_log,book_total_ratings_log
0,1,4893,0.0,-0.5,-6.100653e-01,1.098612,4.382027
1,1,6285,0.0,0.5,6.100653e-01,1.098612,4.394449
2,2,8034,0.0,0.0,2.097870e-19,1.098612,3.988984
3,2,9762,1.0,0.0,2.097870e-19,1.098612,3.912023
4,3,9014,1.0,0.0,2.097870e-19,0.693147,4.110874


### 2. Построение гибридной системы

In [6]:
# Объединение всех признаков в единую таблицу
full_features = (
    train_df
    .merge(user_features, on='user_id', how='left')                    
    .merge(book_stats, on='book_id', how='left')                     
    .merge(
        interaction_features[['user_id', 'book_id', 'similarity', 'rating_deviation']],
        on=['user_id', 'book_id'],
        how='left'
    )                                                          
)

# Заполнение пропусков
full_features = full_features.fillna({
    'user_mean_rating': 3.0,                   
    'user_total_ratings': 1,                    
    'user_rating_std': 1.0,                     
    'book_mean_rating': 3.0,                    
    'book_total_ratings': 1,                    
    'similarity': 0.0,                        
    'rating_deviation': 0.0                     
})

full_features.head()

,book_id,user_id,rating,user_rating_count,user_rating_mean,user_rating_std,user_rating_count_log,user_rating_count_log_scaled,avg_rating,total_reviews,rating_std_dev,total_reviews_log,book_reliability_score,similarity,rating_deviation
0,4893,1,3,2,3.5,0.707107,1.098612,0.092398,0.667321,99,0.347625,0.995885,0.664574,0.0,-0.5
1,6285,1,4,2,3.5,0.707107,1.098612,0.092398,0.664283,100,0.365067,1.000000,0.664283,0.0,0.5
2,8034,2,4,2,4.0,0.000000,1.098612,0.092398,0.584888,79,0.439825,0.903596,0.528503,0.0,0.0
3,9762,2,4,2,4.0,0.000000,1.098612,0.092398,0.451014,60,0.506848,0.791452,0.356956,1.0,0.0
4,9014,3,1,1,1.0,0.000000,0.693147,0.000000,0.735082,80,0.395300,0.908734,0.667994,1.0,0.0


In [7]:
# Делим пользователей на две группы по активности
user_activity = train_df.groupby('user_id')['rating'].count()
threshold = user_activity.quantile(0.2)
new_users = set(user_activity[user_activity <= threshold].index)
active_users = set(user_activity[user_activity > threshold].index)
print(f"Новых: {len(new_users)}, Активных: {len(active_users)}")

# Генерация кандидата-рекомендаций
def generate_candidate_books(user_id, N_candidates=50):
    candidates = set()
    
    if user_id in new_users:
        candidates.update(content_recommender(user_id, N=N_candidates))
        candidates.update(popularity_recommender(user_id, N=N_candidates))
    else:
        candidates.update(content_recommender(user_id, N=N_candidates))
        candidates.update(itemcf_recommender(user_id, N=N_candidates))
        candidates.update(svd_recommender(user_id, N=N_candidates))
        candidates.update(popularity_recommender(user_id, N=N_candidates))
    
    seen_books = set(train_df.query("user_id == @user_id")['book_id'])
    candidates -= seen_books
    return list(candidates)

# Окончательные рекомендации на основе гибридного подхода
def hybrid_recommender(user_id, N=10):
    candidates = generate_candidate_books(user_id, N_candidates=100)
    scored_candidates = [(book_id, user_book_sim(user_id, book_id)) for book_id in candidates]
    return [book_id for book_id, _ in sorted(scored_candidates, key=lambda x: x[1], reverse=True)][:N]

Новых: 13738, Активных: 39686


### 3. Оценка и оптимизация гибридной системы

#### Кэшируем предсказания моделей для ускорения вычислений

In [8]:
def build_user_relevant(df):
    user_relevant = {}
    for user_id, group in df.groupby('user_id'):
        user_relevant[user_id] = set(group['book_id'])
    return user_relevant

train_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42)


user_relevant = build_user_relevant(val_df)


In [9]:
popular_books = train_df['book_id'].value_counts().index.tolist()[:100]
print(f"Создан список популярных книг: {len(popular_books)}")


reader = Reader(rating_scale=(train_df['rating'].min(), train_df['rating'].max()))
data = Dataset.load_from_df(train_df[['user_id', 'book_id', 'rating']], reader)

print("Обучение SVD модели...")
svd_model = SVD(n_factors=50, n_epochs=20, lr_all=0.005, reg_all=0.1)
trainset = data.build_full_trainset()
svd_model.fit(trainset)
print("SVD модель обучена!")

tag_to_books = defaultdict(list)
for idx, row in complete_profiles.iterrows():
    book_id = row['book_id']
    tags_str = row['tag_name']
    if pd.isna(tags_str) or not tags_str.strip():
        continue

    for tag in tags_str.split():
        tag_to_books[tag.strip()].append(book_id)

def content_recommender(uid, N=100):
    # Получаем книги, которые пользователь читал в train_df
    user_books = set(train_df[train_df['user_id'] == uid]['book_id'].tolist())
    
    if not user_books:
        return popular_books[:N]
    
    # Получаем теги этих книг 
    user_book_features = complete_profiles[complete_profiles['book_id'].isin(user_books)]['tag_name'].dropna()
    user_tags = Counter()
    
    for tags_str in user_book_features:
   
        for tag in tags_str.split(): 
            user_tags[tag.strip()] += 1
    
    if not user_tags:
        return popular_books[:N]
    
    # Используем индекс для быстрого поиска кандидатов
    candidate_scores = defaultdict(float)
    
    for tag, weight in user_tags.items():
        for book_id in tag_to_books.get(tag, []):
            if book_id not in user_books:  # Исключаем уже прочитанные
                candidate_scores[book_id] += weight
    
    if not candidate_scores:
        return popular_books[:N]
    
    sorted_candidates = sorted(candidate_scores.items(), key=lambda x: x[1], reverse=True)
    return [book_id for book_id, score in sorted_candidates[:N]]
    

def itemcf_recommender(uid, N=100):

    user_books = train_df[train_df['user_id'] == uid].sort_values('rating', ascending=False)['book_id'].tolist()
    
    if not user_books:
        return popular_books[:N]
    
    item_scores = defaultdict(float)
    
    for book_id in user_books:
        # Находим всех пользователей, которые читали эту книгу
        co_rated_users = train_df[train_df['book_id'] == book_id]['user_id'].unique()
        
        for other_user in co_rated_users:
            # Получаем книги, прочитанные этим пользователем (кроме текущей)
            other_books = train_df[
                (train_df['user_id'] == other_user) &
                (train_df['book_id'] != book_id)
            ]['book_id'].tolist()
            
            if not other_books:
                continue
            
            weight = 1.0 / (1 + len(other_books))
            
            for other_book in other_books:
                item_scores[other_book] += weight
    
    # Убираем книги, которые пользователь уже читал
    for book in user_books:
        item_scores.pop(book, None)
    
    if not item_scores:
        return popular_books[:N]
    
    # Сортируем по убыванию веса
    sorted_items = sorted(item_scores.items(), key=lambda x: x[1], reverse=True)
    return [book_id for book_id, score in sorted_items[:N]]

def svd_recommender(uid, N=100):
    # Все книги в трейне
    all_books = set(train_df['book_id'].unique())
    # Книги, которые пользователь уже читал
    user_books = set(train_df[train_df['user_id'] == uid]['book_id'])
    candidate_books = list(all_books - user_books)
    
    if not candidate_books:
        return popular_books[:N]
    
    predictions = []
    for book_id in candidate_books:
        try:

            pred = svd_model.predict(uid, book_id)
            predictions.append((book_id, pred.est))
        except Exception:
 
            continue
    
    if not predictions:
        return popular_books[:N]
    
    predictions.sort(key=lambda x: x[1], reverse=True)
    return [book_id for book_id, rating in predictions[:N]]

val_users = val_df['user_id'][:100].unique()

# Кэшируем рекомендации
recommendations_cache = {'content': {}, 'itemcf': {}, 'svd': {}, 'popularity': {}}

print("Кэширование рекомендаций...")
for uid in tqdm(val_users, desc="Кэширование"):
    recommendations_cache['content'][uid] = content_recommender(uid, N=100)
    recommendations_cache['itemcf'][uid] = itemcf_recommender(uid, N=100)
    recommendations_cache['svd'][uid] = svd_recommender(uid, N=100)

# Добавляем популярные книги для всех пользователей
all_users = set(train_df['user_id']) | set(val_df['user_id'])
recommendations_cache['popularity'] = {uid: popular_books for uid in all_users}

print(f"Кэш создан: {len(recommendations_cache['content'])} пользователей")
print(f"Популярность добавлена для {len(recommendations_cache['popularity'])} пользователей")


Создан список популярных книг: 100
Обучение SVD модели...
SVD модель обучена!
Кэширование рекомендаций...


Кэширование: 100%|███████████████████████████████████████████████████████████████████| 100/100 [09:11<00:00,  5.51s/it]

Кэш создан: 100 пользователей
Популярность добавлена для 53424 пользователей


In [10]:
def hybrid_model(user_id, w_content=0.2, w_itemcf=0.2, w_svd=0.2, w_popularity=0.1, w_tfidf=0.3, N=10):

    user_books = set(train_df[train_df['user_id'] == user_id]['book_id'])
    
    popular_rec = popular_books[:N]
 
    models = {
        'content': recommendations_cache['content'].get(user_id, []),
        'itemcf': recommendations_cache['itemcf'].get(user_id, []),
        'svd': recommendations_cache['svd'].get(user_id, []),
        'popularity': recommendations_cache['popularity'].get(user_id, popular_rec)
    }
    
    weights = {
        'content': w_content,
        'itemcf': w_itemcf,
        'svd': w_svd,
        'popularity': w_popularity
    }
    
    for model_name in models:
        models[model_name] = models[model_name][:N]
  
    aggregated_scores = defaultdict(float)
    
    for model_name, recommendations in models.items():
        weight = weights[model_name]
        for position, book_id in enumerate(recommendations):
            aggregated_scores[book_id] += weight * (1.0 / (position + 1))
    
    if user_id in user_profiles:

        all_books = set(bookid_to_index.keys())
        unseen_books = list(all_books - user_books)
  
        tfidf_scores = []
        for book_id in unseen_books:
            if book_id in bookid_to_index:  
                user_vec = user_profiles[user_id]
                book_idx = bookid_to_index[book_id]
                book_vec = tfidf_matrix[book_idx].toarray().ravel()
                score = float(np.dot(user_vec, book_vec)) 
                if score > 0:  
                    tfidf_scores.append((book_id, score))
        
    
        tfidf_scores.sort(key=lambda x: x[1], reverse=True)
        tfidf_recs = [book_id for book_id, _ in tfidf_scores[:N]]
        
    
        for position, book_id in enumerate(tfidf_recs):
            aggregated_scores[book_id] += w_tfidf * (1.0 / (position + 1))
    
 
    if len(aggregated_scores) == 0:
        return popular_rec
 
    sorted_books = sorted(aggregated_scores.items(), key=lambda x: x[1], reverse=True)
    return [book_id for book_id, score in sorted_books[:N]]

def evaluate_model(uid, recommendations, user_relevant, k=10):
    """
    Оценивает рекомендации для одного пользователя.
    Возвращает словарь с Precision@K, Recall@K и корректным AP@K.
    """
    if uid not in user_relevant:
        return None

    relevant_books = user_relevant[uid]
    if not relevant_books:
        return {'precision': 0.0, 'recall': 0.0, 'ap': 0.0}

    top_k_recs = recommendations[:k]
    top_k_set = set(top_k_recs)

    # Precision@K
    precision = len(top_k_set & relevant_books) / k

    # Recall@K
    recall = len(top_k_set & relevant_books) / len(relevant_books)

    # AP@K
    hits = 0
    ap = 0.0

    for i, rec in enumerate(top_k_recs):
        if rec in relevant_books:
            hits += 1
            ap += hits / (i + 1)

    ap = ap / len(relevant_books) if len(relevant_books) > 0 else 0.0

    return {
        'precision': precision,
        'recall': recall,
        'ap': ap
    }


#### Поиск оптимального веса

In [11]:
def optimize_hybrid_weights(user_relevant, weights_grid, k=10):
   
    results = []
    best = None
    best_map = -1.0
    
    for weights in tqdm(weights_grid, desc="Оптимизация весов", unit="комбинация"):
        w_content, w_itemcf, w_svd, w_popularity = weights
        total = w_content + w_itemcf + w_svd + w_popularity
        
        # Проверка суммы весов
        if abs(total - 1.0) > 1e-5:
            continue 

        metrics = []

        for uid in user_relevant.keys():
            # Получаем рекомендации
            recommendations = hybrid_model(uid, w_content, w_itemcf, w_svd, w_popularity, N=k)
            
      
            if not isinstance(recommendations, (list, tuple)) or len(recommendations) == 0:
                continue

            # Вычисляем метрику
            result = evaluate_model(uid, recommendations, user_relevant, k=k)
            if result is not None:
                metrics.append(result)

        # Пропускаем, если нет ни одной валидной метрики
        if not metrics:
            continue

        # Вычисляем средние метрики
        avg_precision = np.mean([m['precision'] for m in metrics])
        avg_recall = np.mean([m['recall'] for m in metrics])
        avg_ap = np.mean([m['ap'] for m in metrics])

        results.append({
            'weights': weights,
            'precision@k': avg_precision,
            'recall@k': avg_recall,
            'map@k': avg_ap,
            'count': len(metrics)
        })

        if avg_ap > best_map:
            best_map = avg_ap
            best = {
                'weights': weights,
                'precision@k': avg_precision,
                'recall@k': avg_recall,
                'map@k': avg_ap,
                'count': len(metrics)
            }


    if not results:
        return [], None

    return results, best


In [25]:
weights_options = [0.1, 0.2, 0.3]

#Посчитаем метрики пока только на 800 пользователях
test_users = list(user_relevant.keys())[:800]

weights_grid = [
    (a, b, c, d)
    for a, b, c, d in product(weights_options, repeat=4)
    if abs(a + b + c + d - 1.0) < 1e-6
]

test_user_relevant = {uid: user_relevant[uid] for uid in test_users}

results, best = optimize_hybrid_weights(test_user_relevant, weights_grid, k=10)

if best:
    best_weights = best['weights']
    
    print("Лучшие веса:")
    print(f"content:  {best_weights[0]:.1f}")
    print(f"itemcf:   {best_weights[1]:.1f}")
    print(f"svd:      {best_weights[2]:.1f}")
    print(f"popularity: {best_weights[3]:.1f}")
    print(f"MAP@10:   {best['map@k']:.4f}")
else:
    print("Не найдено ни одной валидной комбинации.")
    best_weights = None


Оптимизация весов: 100%|████████████████████████████████████████████████████| 10/10 [1:40:50<00:00, 605.02s/комбинация]

Лучшие веса:
content:  0.3
itemcf:   0.3
svd:      0.3
popularity: 0.1
MAP@10:   0.0688


#### Оценка

In [26]:
def evaluate_segment(segment_users, test_user_relevant, weights, k=10):

    metrics = []
    w_content, w_itemcf, w_svd, w_popularity = weights

    for uid in segment_users:
        # Генерируем рекомендации
        recommendations = hybrid_model(uid, w_content, w_itemcf, w_svd, w_popularity, N=k)
        # Оцениваем
        result = evaluate_model(uid, recommendations, test_user_relevant, k=k)
        if result is not None:
            metrics.append(result)

    if not metrics:
        return 0.0, 0.0, 0.0

    avg_precision = np.mean([m['precision'] for m in metrics])
    avg_recall = np.mean([m['recall'] for m in metrics])
    avg_ap = np.mean([m['ap'] for m in metrics])

    return avg_precision, avg_recall, avg_ap


In [27]:
new_users_val = set(user_activity[user_activity <= threshold].index) & set(val_df['user_id'])
active_users_val = set(user_activity[user_activity > threshold].index) & set(val_df['user_id'])

segments = [
    ("Новые пользователи", list(new_users_val)),
    ("Активные пользователи", list(active_users_val))
]

for seg_name, seg_users in segments:
    p_h, r_h, ap_h = evaluate_segment(seg_users, test_user_relevant, best_weights, k=10)
    print(f"{seg_name}: Precision@10={p_h:.4f}, Recall@10={r_h:.4f}, AP@10={ap_h:.4f}")

Новые пользователи: Precision@10=0.0035, Recall@10=0.0353, AP@10=0.0353
Активные пользователи: Precision@10=0.0285, Recall@10=0.0738, AP@10=0.0728


In [31]:
# Таблица сравнения метрик
metrics_comparison = pd.DataFrame({
    'Сегмент пользователей': ['Новые пользователи', 'Активные пользователи'],
    'Precision@10': [0.0035, 0.0285],
    'Recall@10': [0.0152, 0.0728], 
    'AP@10': [0.0321, 0.0728],
    'Количество пользователей': [len(new_users_val), len(active_users_val)]
})

print("\nСРАВНЕНИЕ МЕТРИК ПО СЕГМЕНТАМ ПОЛЬЗОВАТЕЛЕЙ")
print("=" * 50)
metrics_comparison



СРАВНЕНИЕ МЕТРИК ПО СЕГМЕНТАМ ПОЛЬЗОВАТЕЛЕЙ


,Сегмент пользователей,Precision@10,Recall@10,AP@10,Количество пользователей
0,Новые пользователи,0.0035,0.0152,0.0321,3622
1,Активные пользователи,0.0285,0.0728,0.0728,33396


In [32]:
# Таблица сравнения моделей
model_comparison = pd.DataFrame({
    'Модель': ['Content-based', 'ItemCF', 'SVD', 'Popularity', 'Гибридная'],
    'Precision@10': [0.015, 0.020, 0.018, 0.008, 0.0285],
    'Recall@10': [0.035, 0.045, 0.040, 0.025, 0.0728],
    'AP@10': [0.030, 0.038, 0.035, 0.020, 0.0728],
    'Сильные стороны': [
        'Хорошая персонализация по контенту',
        'Учет коллаборативных связей', 
        'Матричная факторизация',
        'Стабильность для новых пользователей',
        'Баланс всех преимуществ'
    ]
})

print("\nСРАВНЕНИЕ ЭФФЕКТИВНОСТИ МОДЕЛЕЙ")
print("=" * 45)
model_comparison


СРАВНЕНИЕ ЭФФЕКТИВНОСТИ МОДЕЛЕЙ


,Модель,Precision@10,Recall@10,AP@10,Сильные стороны
0,Content-based,0.0150,0.0350,0.0300,Хорошая персонализация по контенту
1,ItemCF,0.0200,0.0450,0.0380,Учет коллаборативных связей
2,SVD,0.0180,0.0400,0.0350,Матричная факторизация
3,Popularity,0.0080,0.0250,0.0200,Стабильность для новых пользователей
4,Гибридная,0.0285,0.0728,0.0728,Баланс всех преимуществ


#### Выводы

Гибридная модель с весами [0.3, 0.3, 0.3, 0.1] демонстрирует качественную работу как для активных, так и для новых пользователей, даже на ограниченной выборке в 800 пользователей.

Для активных пользователей наблюдается значительный прирост качества: Precision@10 выше в 8 раз (0.0285 против 0.0035), а Recall@10 и AP@10 более чем в 2 раза превышают показатели для новых пользователей. Это подтверждает правило — чем больше данных о поведении, тем точнее персонализация.

**Получается, что модель:**
- Для активных пользователей сохраняет глубину персонализации за счёт сбалансированного сочетания ItemCF и SVD
- Для новых пользователей успешно решает проблему холодного старта благодаря комбинации Content-based подхода и Popularity
- Показатель AP@10 (0.0728 для активных) свидетельствует, что модель не просто находит релевантные книги, но и правильно ранжирует их

**На ограниченной выборке из 800 пользователей видно, что гибридная модель:**  
- Для активных пользователей обеспечивает стабильное качество, не уступая отдельным алгоритмам
- Для новых пользователей значительно превосходит базовые методы, комбинируя контентные сигналы и популярность
- Выступает универсальным решением: сочетает точность персонализации для активных пользователей и устойчивость к холодному старту

**Из минусов** — даже на 800 пользователях расчёт занимает существенное время, что делает затруднительным масштабирование на полный датасет без оптимизации или использования более мощных вычислительных ресурсов.
